# Acquisition du corpus officiel (sans modèle)

Ce notebook montre comment ajouter de nouvelles sources au manifest puis tirer leur HTML officiel depuis EUR-Lex / AMF / Légifrance. **Aucun LLM utilisé** — juste des téléchargements HTML et du parsing DOM.

Sortie : un JSONL de `NormativeUnit` prêt pour `regindex extract` (quand tu seras prêt à relancer le modèle).

In [ ]:
from pathlib import Path

import yaml

from regulatory_index.ingestion.acquire import (
    MANIFEST_PATH,
    acquire_all,
    acquire_one,
    load_manifest,
)
from regulatory_index.ingestion.eurlex_fetcher import eurlex_url, fetch_html
from regulatory_index.ingestion.eurlex_html_parser import parse_articles
from regulatory_index.schemas.sources_registry import load_sources_registry

## 1. Manifest courant

Ce que `regindex acquire` va télécharger.

In [ ]:
for entry in load_manifest():
    print(entry)

## 2. Sources connues du registry

Si tu veux ajouter une entrée au manifest, il faut que son `source_id` existe d'abord dans `config/sources_registry.yaml`.

In [ ]:
registry = load_sources_registry()
for src_id, entry in registry.items():
    print(f'{src_id:<32} L{entry.level:<8} {entry.issuer:<22} {entry.language}  -> {entry.title[:60]}')

## 3. Tester une URL EUR-Lex avant ajout

Donne un CELEX et vois quels articles il contient. Utile pour décider quels filtrer.

In [ ]:
CELEX = '32011L0061'   # change ici pour tester un autre acte
LANG = 'EN'
print(eurlex_url(CELEX, LANG))
html = fetch_html(CELEX, LANG, timeout=30)
print(f'  -> {len(html)} chars HTML')

preview_units = parse_articles(
    html,
    source_id='PREVIEW',
    celex=CELEX,
    language=LANG,
    title='Preview',
    level=1,
    issuer='EU_Parliament_Council',
    url=eurlex_url(CELEX, LANG),
)
for u in preview_units:
    print(f"  Art {u.source_meta['article']:<8} {len(u.text):>6} chars   {u.hierarchy_path[:70]}")

## 4. Étendre le corpus

Modifie `config/sources_manifest.yaml` pour ajouter une nouvelle entrée (ex: une autre série d'articles), puis relance la cellule ci-dessous. Aucun appel LLM, juste téléchargement + parsing.

Quand le PC aura refroidi, lancer `uv run regindex extract data/units/corpus.jsonl` pour extraire les nouvelles unités via Ollama.

In [ ]:
counts = acquire_all()
counts